# Extracción, limpieza y augmentation del dataset AQS-EPA

**Curso:** Programación Concurrente y Distribuida — CC65 — UPC
**Trabajo Parcial — 2026-01**

---

Este notebook ejecuta la **Fase 1** del proyecto: preparar el dataset que después
consumirá la solución concurrente y distribuida en Go.

**Pipeline:**
1. Descarga de 22 archivos `annual_conc_by_monitor_YYYY.zip` (años 2004–2025) desde el AQS-EPA
2. Carga y consolidación en un DataFrame
3. Filtrado por los 4 parámetros de interés (todo Estados Unidos — *Path B*)
4. Limpieza (3 reglas)
5. **Checkpoint**: export del DataFrame limpio en CSV
6. Data augmentation (bootstrap + ruido gaussiano) hasta 3,000,000 de filas
7. Validación pre/post augmentation
8. Export final en CSV

**Fuente:** https://aqs.epa.gov/aqsweb/airdata/download_files.html

## 1. Setup

In [1]:
# Instalación de paquetes (la mayoría ya están en Colab por defecto).
!pip install -q tqdm requests pandas numpy

In [2]:
import os
import io
import zipfile
import time
import shutil
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

print(f'pandas {pd.__version__} | numpy {np.__version__}')

pandas 2.2.2 | numpy 2.0.2


## 2. Configuración

Variables centralizadas. Si necesitas cambiar parámetros, años, o rutas, modifica solo esta celda.

In [3]:
# --- Parámetros AQS (códigos oficiales) ---
POLLUTANTS = {
    '88101': 'PM2.5',
    '42602': 'NO2',
    '44201': 'O3',
    '42101': 'CO',
}
TARGET_PARAM_CODES = list(POLLUTANTS.keys())

# --- Estándar primario por parámetro (el más reciente) ---
# Sirve para deduplicar: cada monitor-año aparece N veces, una por Pollutant Standard.
# Nos quedamos con un único estándar de referencia por parámetro.
PRIMARY_STANDARDS = {
    '88101': 'PM25 Annual 2024',     # PM2.5 anual
    '42602': 'NO2 Annual 1971',      # NO2 anual
    '44201': 'Ozone 8-hour 2015',    # O3 8 horas (estándar más reciente)
    '42101': 'CO 8-hour 1971',       # CO 8 horas
}

# --- Event Types a conservar (descartamos las versiones 'Excluded') ---
KEEP_EVENT_TYPES = {'No Events', 'Events Included'}

# --- Rango temporal ---
YEAR_START = 2004
YEAR_END = 2025  # inclusive  -> 22 años
YEARS = list(range(YEAR_START, YEAR_END + 1))

# --- URL pattern oficial EPA ---
AQS_URL_TEMPLATE = 'https://aqs.epa.gov/aqsweb/airdata/annual_conc_by_monitor_{year}.zip'

# --- Target tamaño dataset final (post augmentation) ---
TARGET_ROWS = 3_000_000

# --- Augmentation: σ relativa para ruido gaussiano por columna ---
# σ_efectivo = NOISE_SIGMA_RATIO * std(columna)
NOISE_SIGMA_RATIO = 0.05  # 5% de la desviación estándar real

# --- Random seed para reproducibilidad ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Parámetros: {list(POLLUTANTS.values())}')
print(f'Años: {YEAR_START}-{YEAR_END} ({len(YEARS)} archivos)')
print(f'Target final: {TARGET_ROWS:,} filas')

Parámetros: ['PM2.5', 'NO2', 'O3', 'CO']
Años: 2004-2025 (22 archivos)
Target final: 3,000,000 filas


## 3. Google Drive (opcional pero recomendado)

Montar Drive permite que los archivos persistan entre sesiones. Si no lo montas,
los archivos quedan en `/content/` y se pierden al cerrar el runtime.

**Si vas a re-ejecutar el notebook**, conviene montar Drive.

In [ ]:
# Si trabajas en Colab, descomenta estas 2 líneas para montar Drive:
from google.colab import drive
drive.mount('/content/drive')


In [5]:

# Carpeta raíz de trabajo. Cambiar a una en tu Drive si lo montaste:
BASE_DIR = Path('/content/drive/MyDrive/aqs_data')

RAW_DIR = BASE_DIR / 'raw'              # zips descargados
CLEAN_CSV = BASE_DIR / 'aqs_clean.csv'  # checkpoint post-limpieza
FINAL_CSV = BASE_DIR / 'aqs_final_3M.csv'  # dataset final con augmentation

for d in (BASE_DIR, RAW_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Directorio de trabajo: {BASE_DIR}')

Directorio de trabajo: /content/drive/MyDrive/aqs_data


## 4. Descarga de archivos Annual Summary

Cada archivo es un zip que contiene un CSV con el resumen anual de TODOS los monitores
y TODOS los parámetros para ese año a nivel nacional.

**Idempotente**: si ya existe el zip en disco, se omite la descarga.

In [6]:
def download_file(url: str, dest: Path, max_retries: int = 3, timeout: int = 120) -> bool:
    """Descarga con reintentos y caché local."""
    if dest.exists() and dest.stat().st_size > 1024:
        return True  # ya descargado

    tmp = dest.with_suffix(dest.suffix + '.part')
    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1 << 16):
                        f.write(chunk)
            tmp.rename(dest)
            return True
        except requests.RequestException as e:
            print(f'  Intento {attempt}/{max_retries} falló: {e}')
            if tmp.exists():
                tmp.unlink()
            if attempt < max_retries:
                time.sleep(5 * attempt)
    return False

In [7]:
# Descargar los 22 archivos
downloaded = []
missing = []
for year in tqdm(YEARS, desc='Descargando'):
    url = AQS_URL_TEMPLATE.format(year=year)
    dest = RAW_DIR / f'annual_conc_by_monitor_{year}.zip'
    ok = download_file(url, dest)
    (downloaded if ok else missing).append(year)

print(f'Descargados/cacheados: {len(downloaded)} archivos')
if missing:
    print(f'⚠ NO se pudieron descargar: {missing}')

total_size_mb = sum(p.stat().st_size for p in RAW_DIR.glob('*.zip')) / 1e6
print(f'Tamaño total en disco: {total_size_mb:.1f} MB')

Descargando:   0%|          | 0/22 [00:00<?, ?it/s]

Descargados/cacheados: 22 archivos
Tamaño total en disco: 110.3 MB


## 5. Carga, consolidación y filtrado

Estrategia para no agotar RAM en Colab: leer cada zip, filtrar inmediatamente a los
4 parámetros de interés (descarta ~95% de las filas), y solo después concatenar.

Esto es **Path B**: todos los monitores de Estados Unidos, no solo las 4 ciudades análogas.
Las 4 ciudades se usarán más adelante como conjunto de validación/análogo a Lima.

In [8]:
# Columnas que SÍ se cargan (descartamos administrativas y redundantes para ahorrar RAM)
USECOLS = [
    'State Code', 'County Code', 'Site Num',
    'Parameter Code', 'POC',
    'Latitude', 'Longitude',
    'Parameter Name', 'Sample Duration', 'Pollutant Standard',
    'Year', 'Units of Measure', 'Event Type',
    'Observation Count', 'Observation Percent', 'Completeness Indicator',
    'Valid Day Count', 'Required Day Count',
    'Exceptional Data Count', 'Null Data Count',
    'Primary Exceedance Count', 'Secondary Exceedance Count',
    'Num Obs Below MDL',
    'Arithmetic Mean', 'Arithmetic Standard Dev',
    '1st Max Value', '2nd Max Value', '3rd Max Value', '4th Max Value',
    '99th Percentile', '98th Percentile', '95th Percentile',
    '90th Percentile', '75th Percentile', '50th Percentile', '10th Percentile',
    'State Name', 'County Name', 'City Name', 'CBSA Name',
]

# Tipos para evitar warnings y pérdida de leading zeros en los códigos
DTYPES = {
    'State Code': str, 'County Code': str, 'Site Num': str,
    'Parameter Code': str,
}

In [9]:
def load_and_filter_year(year: int) -> pd.DataFrame:
    """Lee el zip de un año, filtra a los 4 parámetros, devuelve DataFrame."""
    zip_path = RAW_DIR / f'annual_conc_by_monitor_{year}.zip'
    if not zip_path.exists():
        return pd.DataFrame()

    with zipfile.ZipFile(zip_path) as zf:
        csv_name = next(n for n in zf.namelist() if n.lower().endswith('.csv'))
        with zf.open(csv_name) as fh:
            df = pd.read_csv(fh, usecols=USECOLS, dtype=DTYPES, low_memory=False)

    # Filtro inmediato a los 4 parámetros (reduce tamaño ~20x)
    df = df[df['Parameter Code'].isin(TARGET_PARAM_CODES)].copy()
    return df

# Cargar todos los años
parts = []
for year in tqdm(YEARS, desc='Cargando + filtrando'):
    parts.append(load_and_filter_year(year))

df_raw = pd.concat(parts, ignore_index=True)
del parts
print(f'Filas tras filtro por parámetros: {len(df_raw):,}')
print(f'Memoria del DataFrame: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_raw.head(3)

Cargando + filtrando:   0%|          | 0/22 [00:00<?, ?it/s]

Filas tras filtro por parámetros: 465,902
Memoria del DataFrame: 480.0 MB


,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Parameter Name,Sample Duration,Pollutant Standard,...,98th Percentile,95th Percentile,90th Percentile,75th Percentile,50th Percentile,10th Percentile,State Name,County Name,City Name,CBSA Name
0,01,003,0010,44201,1,30.497478,-87.880258,Ozone,1 HOUR,Ozone 1-hour 1979,...,0.091,0.084,0.075,0.064,0.051,0.033,Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL"
1,01,003,0010,44201,1,30.497478,-87.880258,Ozone,8-HR RUN AVG BEGIN HOUR,Ozone 8-Hour 1997,...,0.078,0.073,0.067,0.057,0.046,0.029,Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL"
2,01,003,0010,44201,1,30.497478,-87.880258,Ozone,8-HR RUN AVG BEGIN HOUR,Ozone 8-Hour 2008,...,0.078,0.073,0.067,0.057,0.046,0.029,Alabama,Baldwin,Fairhope,"Daphne-Fairhope-Foley, AL"


## 6. Limpieza

Tres reglas:

1. **Completeness Indicator = `Y`** — solo monitor-años con datos completos según criterios EPA.
2. **Event Type ∈ {No Events, Events Included}** — descarta variantes 'Excluded' que duplican filas.
3. **Pollutant Standard primario** — cada monitor-año aparece múltiples veces (una por estándar);
   conservamos solo el estándar más reciente por parámetro.

In [10]:
df = df_raw.copy()
print(f'Inicial: {len(df):,} filas')

# Regla 1: Completeness = Y
df = df[df['Completeness Indicator'] == 'Y']
print(f'Tras Completeness=Y: {len(df):,} filas')

# Regla 2: Event Type
df = df[df['Event Type'].isin(KEEP_EVENT_TYPES)]
print(f'Tras Event Type: {len(df):,} filas')

# Regla 3: Pollutant Standard primario por parámetro
primary_mask = df.apply(
    lambda r: r['Pollutant Standard'] == PRIMARY_STANDARDS.get(r['Parameter Code']),
    axis=1,
)
df = df[primary_mask]
print(f'Tras Pollutant Standard primario: {len(df):,} filas')

# Bonus: si todavía quedan duplicados (mismo monitor-año-parametro), promediar
key = ['State Code', 'County Code', 'Site Num', 'POC', 'Parameter Code', 'Year']
before = len(df)
df = df.groupby(key, as_index=False).agg({
    **{c: 'first' for c in ['Latitude', 'Longitude', 'Parameter Name',
                            'Sample Duration', 'Pollutant Standard',
                            'Units of Measure', 'Event Type',
                            'State Name', 'County Name', 'City Name', 'CBSA Name']},
    **{c: 'mean' for c in ['Observation Count', 'Observation Percent',
                            'Valid Day Count', 'Required Day Count',
                            'Exceptional Data Count', 'Null Data Count',
                            'Primary Exceedance Count', 'Secondary Exceedance Count',
                            'Num Obs Below MDL',
                            'Arithmetic Mean', 'Arithmetic Standard Dev',
                            '1st Max Value', '2nd Max Value', '3rd Max Value', '4th Max Value',
                            '99th Percentile', '98th Percentile', '95th Percentile',
                            '90th Percentile', '75th Percentile', '50th Percentile', '10th Percentile']},
})
print(f'Tras dedup por monitor-año-parámetro: {len(df):,} filas (antes: {before:,})')

# Agregar columna legible con el nombre del contaminante
df['pollutant'] = df['Parameter Code'].map(POLLUTANTS)

df.head(3)

Inicial: 465,902 filas
Tras Completeness=Y: 353,059 filas
Tras Event Type: 311,022 filas
Tras Pollutant Standard primario: 62,083 filas
Tras dedup por monitor-año-parámetro: 62,083 filas (antes: 62,083)


,State Code,County Code,Site Num,POC,Parameter Code,Year,Latitude,Longitude,Parameter Name,Sample Duration,...,3rd Max Value,4th Max Value,99th Percentile,98th Percentile,95th Percentile,90th Percentile,75th Percentile,50th Percentile,10th Percentile,pollutant
0,01,003,0010,1,44201,2000,30.497478,-87.880258,Ozone,8-HR RUN AVG BEGIN HOUR,...,0.099,0.097,0.099,0.096,0.085,0.080,0.067,0.055,0.028,O3
1,01,003,0010,1,44201,2004,30.497478,-87.880258,Ozone,8-HR RUN AVG BEGIN HOUR,...,0.081,0.080,0.081,0.078,0.073,0.067,0.057,0.046,0.029,O3
2,01,003,0010,1,44201,2005,30.497478,-87.880258,Ozone,8-HR RUN AVG BEGIN HOUR,...,0.077,0.074,0.077,0.074,0.069,0.065,0.057,0.049,0.023,O3


In [11]:
# Resumen de cobertura por parámetro × año
cobertura = df.groupby(['pollutant', 'Year']).size().unstack(fill_value=0)
print('Filas por contaminante × año:')
cobertura

Filas por contaminante × año:


Year,2000,2004,2005,2006,2007,2008,2009,2011,2012,2013,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
pollutant,,,,,,,,,,,,,,,,,,,,,
CO,536,464,442,428,405,391,362,348,342,330,...,317,305,295,281,263,268,252,248,237,213
NO2,371,381,379,379,361,360,349,337,341,361,...,412,416,408,410,430,417,417,422,412,19
O3,1065,1218,1192,1208,1221,1217,1231,1234,1243,1263,...,1254,1264,1222,1201,1230,1233,1214,1211,1188,374
PM2.5,0,928,893,907,894,829,908,811,836,873,...,1050,1068,1116,1230,1180,1314,1406,1161,1206,0


## 7. ⛳ Checkpoint — Export del dataset limpio

Aquí se guarda el DataFrame **limpio pero sin augmentation**. Esto te permite re-ejecutar
la augmentation con diferentes parámetros sin volver a hacer el download + limpieza.

In [12]:
df.to_csv(CLEAN_CSV, index=False)
size_mb = CLEAN_CSV.stat().st_size / 1e6
print(f'✓ Checkpoint guardado: {CLEAN_CSV}')
print(f'  {len(df):,} filas × {len(df.columns)} columnas | {size_mb:.1f} MB')

✓ Checkpoint guardado: /content/drive/MyDrive/aqs_data/aqs_clean.csv
  62,083 filas × 40 columnas | 18.6 MB


## 8. Data Augmentation

Objetivo: llegar a **3,000,000 filas** combinando dos técnicas:

- **Bootstrap resampling**: muestreo con reemplazo de las filas reales hasta alcanzar el target.
- **Ruido gaussiano** sobre columnas numéricas, con σ proporcional a la std real de cada columna
  (`σ_ruido = NOISE_SIGMA_RATIO × std_columna`).

Se preservan **sin perturbar** las columnas identificadoras (State, County, Site, POC,
Parameter Code, Year) y las categóricas. Solo se perturban las columnas numéricas
(percentiles, means, max values, conteos).

Cada fila lleva una bandera `is_synthetic` para trazabilidad.

In [13]:
# Columnas numéricas a perturbar
NUMERIC_COLS = [
    'Observation Count', 'Observation Percent',
    'Valid Day Count', 'Required Day Count',
    'Exceptional Data Count', 'Null Data Count',
    'Primary Exceedance Count', 'Secondary Exceedance Count',
    'Num Obs Below MDL',
    'Arithmetic Mean', 'Arithmetic Standard Dev',
    '1st Max Value', '2nd Max Value', '3rd Max Value', '4th Max Value',
    '99th Percentile', '98th Percentile', '95th Percentile',
    '90th Percentile', '75th Percentile', '50th Percentile', '10th Percentile',
]

# Columnas que NO se tocan (preserva grouping y categóricas)
PRESERVE_COLS = [c for c in df.columns if c not in NUMERIC_COLS]

# Std real por columna (para calibrar el ruido)
col_stds = df[NUMERIC_COLS].std(numeric_only=True).fillna(0)
noise_sigmas = (col_stds * NOISE_SIGMA_RATIO).values  # vector de σ por columna
print('σ del ruido por columna:')
print(pd.Series(noise_sigmas, index=NUMERIC_COLS).round(4))

σ del ruido por columna:
Observation Count             158.6837
Observation Percent             0.4368
Valid Day Count                 5.1599
Required Day Count              5.3000
Exceptional Data Count         17.7544
Null Data Count                11.6713
Primary Exceedance Count        1.9305
Secondary Exceedance Count      0.9092
Num Obs Below MDL               0.0000
Arithmetic Mean                 0.2655
Arithmetic Standard Dev         0.1883
1st Max Value                   1.4752
2nd Max Value                   1.1995
3rd Max Value                   1.0722
4th Max Value                   0.9981
99th Percentile                 0.9283
98th Percentile                 0.7766
95th Percentile                 0.6097
90th Percentile                 0.4965
75th Percentile                 0.3480
50th Percentile                 0.2323
10th Percentile                 0.1137
dtype: float64


In [14]:
def augment_to_target(df_base: pd.DataFrame, target_rows: int) -> pd.DataFrame:
    """Bootstrap + ruido gaussiano hasta alcanzar target_rows."""
    n_base = len(df_base)
    n_synthetic = max(0, target_rows - n_base)
    print(f'Filas reales: {n_base:,}  →  sintéticas a generar: {n_synthetic:,}')

    if n_synthetic == 0:
        out = df_base.copy()
        out['is_synthetic'] = False
        return out

    # Bootstrap: muestrear con reemplazo
    idx = np.random.randint(0, n_base, size=n_synthetic)
    synth = df_base.iloc[idx].reset_index(drop=True).copy()

    # Ruido gaussiano vectorizado sobre las columnas numéricas
    noise = np.random.normal(
        loc=0.0,
        scale=noise_sigmas,
        size=(n_synthetic, len(NUMERIC_COLS)),
    )
    synth[NUMERIC_COLS] = synth[NUMERIC_COLS].values + noise

    # Constraints físicos: ningún valor numérico puede ser negativo (concentraciones)
    for c in NUMERIC_COLS:
        synth[c] = synth[c].clip(lower=0)

    # Banderas
    df_base = df_base.copy()
    df_base['is_synthetic'] = False
    synth['is_synthetic'] = True

    return pd.concat([df_base, synth], ignore_index=True)

df_final = augment_to_target(df, TARGET_ROWS)
print(f'Tamaño dataset final: {len(df_final):,} filas')
print(f'  Reales:     {(~df_final["is_synthetic"]).sum():,}')
print(f'  Sintéticas: {df_final["is_synthetic"].sum():,}')

Filas reales: 62,083  →  sintéticas a generar: 2,937,917
Tamaño dataset final: 3,000,000 filas
  Reales:     62,083
  Sintéticas: 2,937,917


## 9. Validación pre/post augmentation

Comparamos estadísticas básicas para confirmar que la augmentation no distorsiona
la distribución original más allá del ruido esperado.

In [15]:
real = df_final[~df_final['is_synthetic']]
synth = df_final[df_final['is_synthetic']]

summary_cols = ['Arithmetic Mean', '99th Percentile', '50th Percentile', '1st Max Value']
comp = pd.concat({
    'Real':      real[summary_cols].describe().loc[['mean', 'std', '50%']],
    'Sintética': synth[summary_cols].describe().loc[['mean', 'std', '50%']],
}, axis=1).round(4)
print('Comparación de estadísticas (real vs sintética):')
comp

Comparación de estadísticas (real vs sintética):


Real                                                \
     Arithmetic Mean 99th Percentile 50th Percentile 1st Max Value   
mean          4.2983         13.8053          3.6619       19.7072   
std           5.3100         18.5655          4.6468       29.5031   
50%           0.4301          1.3000          0.4000        2.1000   

           Sintética                                                
     Arithmetic Mean 99th Percentile 50th Percentile 1st Max Value  
mean          4.3386         13.9560          3.6972       19.9504  
std           5.2824         18.4223          4.6246       29.2965  
50%           0.5892          1.9180          0.5110        3.0933

In [16]:
# Cobertura por contaminante en el dataset final
print('Distribución por contaminante en dataset final:')
df_final.groupby('pollutant').agg(
    n_total=('Year', 'size'),
    n_real=('is_synthetic', lambda s: (~s).sum()),
    n_synth=('is_synthetic', 'sum'),
)

Distribución por contaminante en dataset final:


,n_total,n_real,n_synth
pollutant,,,
CO,356152,7381,348771
NO2,394647,8169,386478
O3,1255289,25982,1229307
PM2.5,993912,20551,973361


## 10. Export final

Dataset final en CSV. Este es el archivo que consumirá la solución concurrente y distribuida en Go.

In [17]:
df_final.to_csv(FINAL_CSV, index=False)
size_mb = FINAL_CSV.stat().st_size / 1e6
print(f'✓ Dataset final guardado: {FINAL_CSV}')
print(f'  {len(df_final):,} filas × {len(df_final.columns)} columnas | {size_mb:.1f} MB')

✓ Dataset final guardado: /content/drive/MyDrive/aqs_data/aqs_final_3M.csv
  3,000,000 filas × 41 columnas | 1536.1 MB
